# Nghiên cứu độ phân giải 96³ vs 128³ cho bài toán **phân loại 3D** (glaucoma)

Mục tiêu: chốt hạ **96³ hay 128³** bằng số liệu định lượng ở 4 tầng (Ảnh / Đặc trưng / Nhiệm vụ /
Tài nguyên), theo 4 thí nghiệm:

| # | Thí nghiệm | Đo đạc | Ngưỡng an toàn cho 96³ |
|---|---|---|---|
| 1 | Reconstruction & Interpolation | PSNR, SSIM(3D) sau hạ-mẫu rồi nâng lại | SSIM > 0.85, PSNR > 30 dB |
| 2 | Proxy Model Training | AUC-ROC/PR, ECE, F1, balanced acc | AUC tụt < 1.5% so với 128³ |
| 3 | Feature Space (CKA + Silhouette) | CKA(128,96), sụt Silhouette | CKA > 0.85, sụt < 10% |
| 4 | Batch-size Trade-off | AUC(128³, BS nhỏ) vs AUC(96³, BS lớn) | 96³ + BS lớn ≥ 128³ + BS nhỏ |

**Dữ liệu:** đặt vào `data/glaucoma_all/` (xem `data/README.md`). Notebook **không** tải/không ghi đè dữ liệu.
Raw giữ 200³ (`STORE_RES=200`), resize on-the-fly (`MODEL_RES`).

Log wandb + copy figure/report/model lên Drive `MyDrive/MasterBKDN/Thesis/resolution_96_128[_figures]`.

## 1. Setup + import

In [ ]:
!git clone --depth 1 https://github.com/Tqhuyen/glaucoma-thesis.git /content/glaucoma-thesis 2>/dev/null || git -C /content/glaucoma-thesis pull --ff-only 2>/dev/null || true
%cd /content/glaucoma-thesis
!pip install -q wandb scikit-image scipy scikit-learn matplotlib pandas requests
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
import os, sys, json, time, math

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, "scripts")
import resolution_study as rs

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    print("[warn] không có GPU: proxy training sẽ rất chậm; nên bật runtime GPU")


## 2. Cấu hình

- `STORE_RES = 200` (raw) và `MODEL_RES` (kích thước đưa vào model) là **hai tên khác nhau** — tránh nhầm.
- `RES_LIST = (128, 96)`: hai ứng viên cần so với baseline 200.
- Bật/tắt từng thí nghiệm bằng `RUN_EXP1..4`.
- `PROXY_*`: cấu hình proxy model (mặc định nhẹ để chạy vài giờ).

In [ ]:
SEED = 42
STORE_RES = 200
RES_LIST = (128, 96)
BASELINE_RES = 200

DATA_CANDIDATES = [os.environ.get("GF_DATA_DIR"), "data/glaucoma_all", "glaucoma_all"]
SPLIT = "Training"
VAL_SPLIT = "Validation"

RUN_EXP1 = True
RUN_EXP2 = True
RUN_EXP3 = True
RUN_EXP4 = True

N_EXP1 = 50
PROXY_TRAIN_N = 300
PROXY_VAL_N = 150
PROXY_EPOCHS = 8
PROXY_BS = 2
PROXY_LR = 1e-3
EXP4_BS_LARGE = 6

np.random.seed(SEED)
torch.manual_seed(SEED)
FIG_DIR = os.path.join("figures", "resolution")
os.makedirs(FIG_DIR, exist_ok=True)
print("config:", "store", STORE_RES, "| candidates", RES_LIST, "| baseline", BASELINE_RES)


## 3. Nạp dữ liệu

Tìm `Training/Validation_{volumes,labels}.npy` trong `data/glaucoma_all/` (hoặc `GF_DATA_DIR`).
Chuẩn hoá shape về `(N, 200, 200, 200)` uint8. Nếu thiếu file -> báo rõ và dừng.

In [ ]:
def find_data_dir():
    for c in DATA_CANDIDATES:
        if c and os.path.isfile(os.path.join(c, f"{SPLIT}_volumes.npy")):
            return c
    return None


DATA_DIR = find_data_dir()
if DATA_DIR is None:
    raise RuntimeError(
        "Không thấy dữ liệu. Đặt file vào data/glaucoma_all/ gồm "
        f"{SPLIT}_volumes.npy, {SPLIT}_labels.npy, {VAL_SPLIT}_volumes.npy, {VAL_SPLIT}_labels.npy "
        "(xem data/README.md), hoặc đặt env GF_DATA_DIR."
    )
print("[data] dir =", DATA_DIR)


def load_split(split):
    vol_path = os.path.join(DATA_DIR, f"{split}_volumes.npy")
    lab_path = os.path.join(DATA_DIR, f"{split}_labels.npy")
    vols = np.load(vol_path, mmap_mode="r")
    labels = np.load(lab_path)
    assert vols.shape[1:] in ((STORE_RES,) * 3, (1, STORE_RES, STORE_RES, STORE_RES)), vols.shape
    return vols, labels


def get_vol(vols, j):
    v = np.asarray(vols[j])
    return v[0] if v.ndim == 4 else v


TRAIN_VOLS, TRAIN_LABELS = load_split(SPLIT)
VAL_VOLS, VAL_LABELS = load_split(VAL_SPLIT)
print(f"[data] train {TRAIN_VOLS.shape} pos={int(np.sum(TRAIN_LABELS))} | "
      f"val {VAL_VOLS.shape} pos={int(np.sum(VAL_LABELS))}", flush=True)


## 4. Dataset + proxy model + hàm train

`NumpyVolumeDataset` resize on-the-fly 200 -> `MODEL_RES`, chuẩn hoá `/255`.
Proxy model = 3D CNN residual nhẹ (GroupNorm vì batch nhỏ). Dùng chung cho mọi kích thước để so sánh công bằng.

In [ ]:
class NumpyVolumeDataset(Dataset):
    def __init__(self, vols, labels, model_res, n_max=0, seed=0):
        self.vols = vols
        self.labels = np.asarray(labels)
        self.model_res = model_res
        idx = np.arange(len(self.labels))
        if n_max and n_max < len(idx):
            idx = np.random.default_rng(seed).choice(idx, size=n_max, replace=False)
        self.idx = idx

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = int(self.idx[i])
        v = get_vol(self.vols, j)
        x = rs.resize_volume(v, (self.model_res,) * 3) / 255.0
        return torch.from_numpy(x)[None].float(), int(self.labels[j])


class Block3D(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv3d(cin, cout, 3, stride, 1, bias=False)
        self.n1 = nn.GroupNorm(8, cout)
        self.conv2 = nn.Conv3d(cout, cout, 3, 1, 1, bias=False)
        self.n2 = nn.GroupNorm(8, cout)
        self.down = None
        if stride != 1 or cin != cout:
            self.down = nn.Sequential(nn.Conv3d(cin, cout, 1, stride, bias=False), nn.GroupNorm(8, cout))

    def forward(self, x):
        idt = x if self.down is None else self.down(x)
        h = F.relu(self.n1(self.conv1(x)), inplace=True)
        h = self.n2(self.conv2(h))
        return F.relu(h + idt, inplace=True)


class Proxy3DCNN(nn.Module):
    def __init__(self, num_classes=2, width=24):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv3d(1, width, 3, 2, 1, bias=False), nn.GroupNorm(8, width),
                                  nn.ReLU(inplace=True), nn.MaxPool3d(2))
        chans = [width, width * 2, width * 4, width * 8]
        self.stage1 = Block3D(chans[0], chans[1], stride=2)
        self.stage2 = Block3D(chans[1], chans[2], stride=2)
        self.stage3 = Block3D(chans[2], chans[3], stride=2)
        self.norm = nn.GroupNorm(8, chans[3])
        self.head = nn.Linear(chans[3], num_classes)

    def embed(self, x):
        h = self.stage3(self.stage2(self.stage1(self.stem(x))))
        h = F.relu(self.norm(h), inplace=True)
        return F.adaptive_avg_pool3d(h, 1).flatten(1)

    def forward(self, x):
        return self.head(self.embed(x))


@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    ps, ys = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
            logits = model(x)
        ps.append(torch.softmax(logits.float(), 1)[:, 1].cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(ps), np.concatenate(ys)


def make_loaders(model_res, bs, n_train, n_val):
    tr = NumpyVolumeDataset(TRAIN_VOLS, TRAIN_LABELS, model_res, n_max=n_train, seed=SEED)
    va = NumpyVolumeDataset(VAL_VOLS, VAL_LABELS, model_res, n_max=n_val, seed=SEED + 1)
    return (DataLoader(tr, batch_size=bs, shuffle=True, num_workers=2, pin_memory=True, drop_last=True),
            DataLoader(va, batch_size=bs, shuffle=False, num_workers=2, pin_memory=True))


def train_proxy(model_res, bs, epochs=PROXY_EPOCHS, tag=""):
    tr, va = make_loaders(model_res, bs, PROXY_TRAIN_N, PROXY_VAL_N)
    model = Proxy3DCNN().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=PROXY_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs * len(tr)))
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    ce = nn.CrossEntropyLoss()
    hist = []
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        losses = []
        for x, y in tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
                loss = ce(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            losses.append(float(loss))
        probs, ys = predict_probs(model, va)
        m = rs.classification_metrics(probs, ys)
        m["loss"] = float(np.mean(losses))
        m["epoch"] = ep + 1
        hist.append(m)
        print(f"[train {tag}] res={model_res} bs={bs} ep {ep+1}/{epochs} loss={m['loss']:.4f} "
              f"auc={m['auc_roc']:.4f} f1={m['f1']:.4f} ece={m['ece']:.4f}", flush=True)
        if globals().get("RUN_WANDB"):
            run.log({f"{tag}/res{model_res}_bs{bs}/loss": m["loss"], f"{tag}/res{model_res}_bs{bs}/auc": m["auc_roc"]}, step=ep + 1)
    probs, ys = predict_probs(model, va)
    final = rs.classification_metrics(probs, ys)
    final["train_min"] = (time.time() - t0) / 60.0
    return model, final, hist


## 5. Wandb + Drive helpers

In [ ]:
if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception as e:
        print("no WANDB_API_KEY:", e)

RUN_NAME = "resolution_96_128_" + time.strftime("%Y%m%d_%H%M%S")
run = None
if os.environ.get("WANDB_API_KEY"):
    try:
        import wandb
        run = wandb.init(project="glaucoma-thesis", name=RUN_NAME,
                         config={"store_res": STORE_RES, "res_list": list(RES_LIST),
                                 "proxy_epochs": PROXY_EPOCHS, "proxy_bs": PROXY_BS})
    except Exception as e:
        print("[wandb] init failed:", e)
RUN_WANDB = run is not None
print("wandb:", RUN_NAME if RUN_WANDB else None)

DRIVE_ROOT = os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/MasterBKDN/Thesis")


def mount_drive():
    if os.path.isdir(DRIVE_ROOT):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return os.path.isdir(DRIVE_ROOT)
    except Exception as e:
        print("[drive] mount skipped:", e)
        return False


DRIVE_FIG = os.path.join(DRIVE_ROOT, "resolution_96_128_figures")
DRIVE_MODEL = os.path.join(DRIVE_ROOT, "resolution_96_128")


## 6. Thí nghiệm 1 — Reconstruction & Interpolation

Lấy `N_EXP1` volume 200³, hạ mẫu xuống 128³/96³ (Gaussian smoothing + trilinear), rồi nâng ngược về 200³.
Tính PSNR/SSIM(3D) so với ảnh gốc. Lưu bảng + ảnh mặt cắt minh hoạ.

In [ ]:
exp1 = {}
if RUN_EXP1:
    n = min(N_EXP1, len(TRAIN_VOLS))
    idx = np.random.default_rng(0).choice(len(TRAIN_VOLS), size=n, replace=False)
    for res in RES_LIST:
        ps, ss = [], []
        for j in idx:
            v = get_vol(TRAIN_VOLS, int(j)).astype(np.float32)
            d = rs.downsample_volume(v, (res,) * 3, mode="gaussian_trilinear")
            u = rs.resize_volume(d, (STORE_RES,) * 3)
            ps.append(rs.psnr(v, u))
            ss.append(rs.ssim3d(v, u))
        exp1[res] = {"psnr": float(np.mean(ps)), "ssim": float(np.mean(ss))}
        print(f"[exp1] {res}^3 -> PSNR {exp1[res]['psnr']:.2f} dB | SSIM {exp1[res]['ssim']:.4f}")

    j = int(idx[0])
    v = get_vol(TRAIN_VOLS, j).astype(np.float32)
    mid = STORE_RES // 2
    panels = [("200^3 (gốc)", v[:, :, mid])]
    for res in RES_LIST:
        d = rs.downsample_volume(v, (res,) * 3, mode="gaussian_trilinear")
        u = rs.resize_volume(d, (STORE_RES,) * 3)
        panels.append((f"{res}^3 -> up", u[:, :, mid]))
    fig, axs = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.4))
    for ax, (t, im) in zip(np.atleast_1d(axs), panels):
        ax.imshow(im, cmap="gray", vmin=np.percentile(v, 1), vmax=np.percentile(v, 99))
        ax.set_title(t, fontsize=10)
        ax.axis("off")
    fig.suptitle("Downsample -> upsample (mặt cắt giữa)")
    p1 = os.path.join(FIG_DIR, "exp1_downsample_upsample.png")
    fig.tight_layout()
    fig.savefig(p1, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print("wrote", p1)
    if RUN_WANDB:
        import wandb
        for res, d in exp1.items():
            run.log({f"exp1/{res}/psnr": d["psnr"], f"exp1/{res}/ssim": d["ssim"]}, step=1)
        run.log({"exp1/img": wandb.Image(p1)}, step=1)


## 7. Thí nghiệm 2 — Proxy Model Training (200/128/96, cùng seed & hyperparams)

In [ ]:
exp2 = {}
models = {}
histories = {}
if RUN_EXP2:
    for res in (BASELINE_RES,) + tuple(RES_LIST):
        model, final, hist = train_proxy(res, PROXY_BS, epochs=PROXY_EPOCHS, tag="exp2")
        exp2[res] = final
        models[res] = model
        histories[res] = hist
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.4))
    for res, hist in histories.items():
        axs[0].plot([h["epoch"] for h in hist], [h["loss"] for h in hist], label=f"{res}^3")
        axs[1].plot([h["epoch"] for h in hist], [h["auc_roc"] for h in hist], label=f"{res}^3")
    axs[0].set_title("Val loss"); axs[0].set_xlabel("epoch")
    axs[1].set_title("Val AUC-ROC"); axs[1].set_xlabel("epoch")
    for a in axs:
        a.legend(); a.grid(alpha=0.3)
    p2 = os.path.join(FIG_DIR, "exp2_learning_curves.png")
    fig.tight_layout(); fig.savefig(p2, dpi=160, bbox_inches="tight"); plt.close(fig)
    print("wrote", p2)
    for res, m in exp2.items():
        print(f"[exp2] {res}^3 AUC={m['auc_roc']:.4f} AP={m['auc_pr']:.4f} F1={m['f1']:.4f} "
              f"bAcc={m['balanced_acc']:.4f} ECE={m['ece']:.4f} ({m['train_min']:.1f} min)")
    if RUN_WANDB:
        import wandb
        for res, m in exp2.items():
            run.log({f"exp2/{res}/{k}": v for k, v in m.items()}, step=2)
        run.log({"exp2/img": wandb.Image(p2)}, step=2)


## 8. Thí nghiệm 3 — Feature Space (CKA + Silhouette)

Lấy vector GAP (bỏ FC cuối) của model 128³ và 96³ trên cùng tập val -> CKA (linear).
Giảm chiều bằng t-SNE -> Silhouette, đo mức sụt của 96³ so với 128³.

In [ ]:
exp3 = {}
if RUN_EXP3:
    feats = {}
    for res in RES_LIST:
        if res not in models:
            print(f"[exp3] thiếu model {res} (cần RUN_EXP2)"); continue
        va = NumpyVolumeDataset(VAL_VOLS, VAL_LABELS, res, n_max=PROXY_VAL_N, seed=SEED + 1)
        dl = DataLoader(va, batch_size=PROXY_BS, shuffle=False, num_workers=2)
        model = models[res].eval()
        fs, ys = [], []
        with torch.no_grad():
            for x, y in dl:
                x = x.to(DEVICE)
                with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
                    f = model.embed(x)
                fs.append(f.float().cpu().numpy()); ys.append(y.numpy())
        feats[res] = np.concatenate(fs); VAL_Y = np.concatenate(ys)
    if 128 in feats and 96 in feats:
        cka = rs.linear_cka(feats[128], feats[96])
        exp3["cka"] = float(cka)
        sil = {}
        from sklearn.manifold import TSNE
        emb2 = {}
        for res in RES_LIST:
            emb = TSNE(n_components=2, perplexity=min(30, len(feats[res]) - 1), init="pca",
                       random_state=0).fit_transform(feats[res])
            emb2[res] = emb
            sil[res] = rs.silhouette_np(feats[res], VAL_Y)
        exp3["silhouette"] = sil
        exp3["silhouette_drop"] = float((sil[128] - sil[96]) / sil[128]) if sil[128] else float("nan")
        fig, axs = plt.subplots(1, 2, figsize=(11, 4.8))
        for ax, res in zip(axs, RES_LIST):
            sc = ax.scatter(emb2[res][:, 0], emb2[res][:, 1], c=VAL_Y, s=8, cmap="coolwarm", alpha=0.7)
            ax.set_title(f"{res}^3 | silhouette={sil[res]:.3f}")
            ax.axis("off")
        fig.suptitle(f"CKA(128,96) = {cka:.3f}")
        p3 = os.path.join(FIG_DIR, "exp3_features_tsne.png")
        fig.tight_layout(); fig.savefig(p3, dpi=160, bbox_inches="tight"); plt.close(fig)
        print(f"[exp3] CKA={cka:.4f} | sil128={sil[128]:.4f} sil96={sil[96]:.4f} "
              f"drop={exp3['silhouette_drop']*100:.2f}%")
        print("wrote", p3)
        if RUN_WANDB:
            import wandb
            run.log({"exp3/cka": cka, "exp3/sil128": sil[128], "exp3/sil96": sil[96],
                     "exp3/sil_drop": exp3["silhouette_drop"], "exp3/img": wandb.Image(p3)}, step=3)


## 9. Thí nghiệm 4 — Batch-size Trade-off (quan trọng nhất)

128³ với BS nhỏ (`PROXY_BS`) vs 96³ với BS lớn (`EXP4_BS_LARGE`). So AUC.

In [ ]:
exp4 = {}
if RUN_EXP4:
    m128, f128, _ = train_proxy(128, PROXY_BS, epochs=PROXY_EPOCHS, tag="exp4")
    m96, f96, _ = train_proxy(96, EXP4_BS_LARGE, epochs=PROXY_EPOCHS, tag="exp4")
    exp4 = {"128_bs%d" % PROXY_BS: f128, "96_bs%d" % EXP4_BS_LARGE: f96}
    print(f"[exp4] 128^3 BS={PROXY_BS}: AUC={f128['auc_roc']:.4f} | "
          f"96^3 BS={EXP4_BS_LARGE}: AUC={f96['auc_roc']:.4f}")
    if RUN_WANDB:
        run.log({"exp4/auc_128_smallbs": f128["auc_roc"], "exp4/auc_96_largebs": f96["auc_roc"]}, step=4)


## 10. Ma trận quyết định + báo cáo

Tổng hợp theo ngưỡng trong đề bài và in khuyến nghị. Ghi JSON + copy lên Drive.

In [ ]:
report = {"exp1": exp1, "exp2": exp2, "exp3": exp3, "exp4": exp4}
checks = {}
if 96 in exp1:
    checks["exp1_ssim>0.85"] = bool(exp1[96]["ssim"] > 0.85)
    checks["exp1_psnr>30"] = bool(exp1[96]["psnr"] > 30.0)
if 128 in exp2 and 96 in exp2:
    checks["exp2_auc_drop<1.5%"] = bool((exp2[128]["auc_roc"] - exp2[96]["auc_roc"]) < 0.015)
if exp3.get("cka") is not None:
    checks["exp3_cka>0.85"] = bool(exp3["cka"] > 0.85)
    checks["exp3_sil_drop<10%"] = bool(exp3["silhouette_drop"] < 0.10)
if exp4:
    k96 = "96_bs%d" % EXP4_BS_LARGE
    k128 = "128_bs%d" % PROXY_BS
    if k96 in exp4 and k128 in exp4:
        checks["exp4_96largebs>=128smallbs"] = bool(exp4[k96]["auc_roc"] >= exp4[k128]["auc_roc"] - 0.005)

report["checks"] = checks
passed = sum(1 for v in checks.values() if v)
report["recommendation"] = ("CHỌN 96^3" if passed >= max(3, len(checks) - 1) and checks.get("exp4_96largebs>=128smallbs", False)
                            else "GIỮ 128^3")
report["passed"] = f"{passed}/{len(checks)}"
print("\n===== KẾT LUẬN =====")
for k, v in checks.items():
    print(("  PASS " if v else "  FAIL ") + k)
print("=> KHUYẾN NGHỊ:", report["recommendation"], f"({report['passed']})")

report_path = os.path.join(FIG_DIR, "resolution_96_vs_128_report.json")
with open(report_path, "w") as fh:
    json.dump(report, fh, indent=2, default=float)
print("wrote", report_path)
if RUN_WANDB:
    run.summary.update({"recommendation": report["recommendation"], "passed": report["passed"]})
    for k, v in checks.items():
        run.summary.update({f"check/{k}": bool(v)})

if mount_drive():
    import shutil
    os.makedirs(DRIVE_FIG, exist_ok=True)
    os.makedirs(DRIVE_MODEL, exist_ok=True)
    for f in os.listdir(FIG_DIR):
        shutil.copy2(os.path.join(FIG_DIR, f), os.path.join(DRIVE_FIG, f))
    for res, model in models.items():
        torch.save({"state_dict": model.state_dict(), "res": res}, os.path.join(DRIVE_MODEL, f"proxy_{res}.pt"))
    print("[drive] figures ->", DRIVE_FIG, "| models ->", DRIVE_MODEL)
else:
    print("[drive] SKIP")

if RUN_WANDB:
    run.finish()
print("done")


## 11. Gợi ý nếu chọn 96³ (Pro-tips)

1. **2.5D / MPR**: cắt 96³ thành 3 lát 2D (Axial/Coronal/Sagittal) 96×96 cho 3 nhánh CNN 2D rồi concat feature.
2. **Attention/SE**: chèn SE hoặc spatial attention để model tự tập trung voxel quan trọng, bỏ qua nền mờ.
3. **Smart downsampling**: dùng Gaussian trước khi hạ mẫu (đã mặc định trong `resolution_study.downsample_volume`) hoặc Max-Pooling 3D cho đặc trưng điểm sáng.

**Cách đọc kết quả:** chỉ chốt 96³ khi `exp4_96largebs>=128smallbs` đúng và đa số check còn lại PASS;
nếu `exp2_auc_drop` > 2% hoặc `exp3_cka` < 0.80 thì giữ 128³ bất kể tài nguyên.
